# Qwen3-Embedding-4B on AWS Trainium via PyTorch Native

This notebook walks through running the [Qwen/Qwen3-Embedding-4B](https://huggingface.co/Qwen/Qwen3-Embedding-4B) model on AWS Trainium using **PyTorch Native (TorchNeuron)** -- the native PyTorch backend for Neuron hardware.

## What You'll Learn

1. How decoder-only LLMs produce embeddings (it's not what you'd expect)
2. The standard HuggingFace code for generating embeddings on CPU/GPU
3. Every modification needed to run on Neuron -- and **why** each change is required
4. How to validate embedding accuracy against a CPU reference
5. How to benchmark throughput on Trainium

## Prerequisites

- AWS Trainium instance (trn2.3xlarge or larger)
- PyTorch Native Beta 2 environment (see [setup guide](https://awsdocs-neuron.readthedocs-hosted.com/en/latest/frameworks/torch/pytorch-native-overview.html))
- `transformers >= 4.51.0` (required for `qwen3` model type)

## Instance Setup

This notebook assumes you're running on a trn2.3xlarge with the PyTorch Native Beta 2 venv:

```bash
# Activate the PyTorch Native venv (created during setup)
source ~/workspace/native_venv/bin/activate

# Install dependencies
pip install transformers>=4.51.0 jupyter
```

---
## Part 1: Understanding the Model

### A Decoder-Only Embedding Model?

Qwen3-Embedding-4B is architecturally **Qwen3ForCausalLM** -- the same decoder-only autoregressive transformer used for text generation. This is surprising because traditionally, embedding models were encoder-only (like BERT or SentenceTransformers).

The key insight: **any language model's hidden states contain rich semantic representations**. By fine-tuning a decoder-only LLM and extracting the hidden state at the right position, you get excellent embeddings.

### How It Produces Embeddings

1. **Input text** is tokenized and fed through all 36 transformer layers
2. **Last-token pooling**: the hidden state at the *last token position* is extracted
3. **L2 normalization**: the vector is normalized to unit length

Why last-token? In a decoder-only model with causal attention, the last token has attended to ALL previous tokens. Its hidden state is a summary of the entire input -- similar to how a `[CLS]` token works in BERT, but emerging from the causal attention pattern.

### Architecture Quick Reference

| Property | Value | Notes |
|----------|-------|-------|
| Architecture | Qwen3ForCausalLM | Decoder-only transformer |
| Parameters | 4B | 36 layers, hidden_size=2560 |
| Attention | GQA 32Q/8KV | 4:1 ratio, head_dim=128 |
| Activation | SiLU + RMSNorm | Standard modern LLM |
| Position encoding | RoPE (theta=1M) | No sliding window |
| Dtype | bfloat16 | Native compute type |
| Embedding dim | 2560 | Supports MRL truncation to 32-2560 |

---
## Part 2: Standard CPU/GPU Code

Let's first look at how you'd normally run this model on CPU or GPU. This is the reference implementation from the HuggingFace model card.

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import math


def last_token_pool(last_hidden_states, attention_mask):
    """Extract the hidden state of the last real token in each sequence.
    
    With left-padding: the last position is always a real token, so just take [:, -1].
    With right-padding: find the actual last non-pad token per sequence.
    """
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[
            torch.arange(batch_size, device=last_hidden_states.device),
            sequence_lengths,
        ]


def pad_to_multiple(tokenizer, texts, multiple=32, max_length=512):
    """Tokenize and pad to the next multiple of `multiple`.
    
    Why: bfloat16 attention on Neuron can produce NaN at certain batch/seq_len
    combinations (e.g., bs=5, seq_len=27). Padding to multiples of 32 avoids this.
    This is a known numerical stability pattern for hardware accelerators.
    """
    # First pass: find the natural padded length
    batch = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    seq_len = batch['input_ids'].shape[1]
    # Round up to next multiple
    padded_len = math.ceil(seq_len / multiple) * multiple
    if padded_len != seq_len:
        # Re-tokenize with explicit max_length to get the rounded padding
        batch = tokenizer(texts, padding='max_length', truncation=True,
                          max_length=padded_len, return_tensors='pt')
    return batch

In [2]:
# Test data: queries (with instruction prefix) and documents (raw text)
queries = [
    "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: What is the capital of France?",
    "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: How does photosynthesis work?",
]

documents = [
    "Paris is the capital and most populous city of France, situated on the Seine River.",
    "Photosynthesis is the process by which green plants convert sunlight into chemical energy, using carbon dioxide and water to produce glucose and oxygen.",
    "The Eiffel Tower was built in 1889 for the World's Fair in Paris, France.",
]

all_texts = queries + documents
print(f"Total texts: {len(all_texts)} ({len(queries)} queries + {len(documents)} documents)")

Total texts: 5 (2 queries + 3 documents)


### CPU Reference Embeddings

We generate embeddings on CPU first. These serve as our ground truth for validating Neuron accuracy.

In [3]:
# Load model on CPU for reference
model_name = "Qwen/Qwen3-Embedding-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

# Note: on CPU we load in float32 for maximum precision reference
cpu_model = AutoModel.from_pretrained(model_name, dtype=torch.float32)
cpu_model.eval()
print(f"Model loaded on CPU: {sum(p.numel() for p in cpu_model.parameters()) / 1e9:.1f}B parameters")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded on CPU: 4.0B parameters


In [4]:
# Tokenize -- pad to multiple of 32 for Neuron compatibility (see Change #5 below)
batch = pad_to_multiple(tokenizer, all_texts, multiple=32)
print(f"Input shape: {batch['input_ids'].shape} (padded to multiple of 32)")

# Generate CPU reference embeddings
with torch.no_grad():
    outputs = cpu_model(**batch)
    cpu_embeddings = last_token_pool(outputs.last_hidden_state, batch['attention_mask'])
    cpu_embeddings = F.normalize(cpu_embeddings, p=2, dim=1)

print(f"CPU embedding shape: {cpu_embeddings.shape}")
print(f"CPU embedding dtype: {cpu_embeddings.dtype}")

# Quick sanity check: query-document similarity
sim_matrix = cpu_embeddings[:len(queries)] @ cpu_embeddings[len(queries):].T
print(f"\nQuery-Document similarity matrix (CPU reference):")
print(sim_matrix.numpy().round(4))
print("\nExpected: high similarity on diagonal (query matches its document)")

Input shape: torch.Size([5, 32]) (padded to multiple of 32)


CPU embedding shape: torch.Size([5, 2560])
CPU embedding dtype: torch.float32

Query-Document similarity matrix (CPU reference):
[[0.7007 0.1502 0.4043]
 [0.1221 0.5007 0.118 ]]

Expected: high similarity on diagonal (query matches its document)


In [5]:
# Save CPU reference for later comparison
cpu_embeddings_saved = cpu_embeddings.clone()

# Free CPU model memory -- we'll load a new one for Neuron
del cpu_model
import gc; gc.collect()
print("CPU reference saved. Model freed.")

CPU reference saved. Model freed.


---
## Part 3: Moving to Neuron -- What Changes and Why

Now we'll load and run the same model on Neuron. Each change is explained.

### Change #1: Device = `"neuron"`

On GPU you'd use `model.to("cuda")`. PyTorch Native registers Neuron as a first-class PyTorch device:

```python
device = torch.device("neuron")  # Instead of "cuda" or "cpu"
```

**Why this works**: PyTorch Native implements the `PrivateUse1` device backend, which PyTorch exposes as `"neuron"`. All standard PyTorch operations (`torch.matmul`, `F.linear`, etc.) are dispatched to NeuronCore hardware through this backend. No special tracing or compilation step needed -- it's true eager mode.

### Change #2: `attn_implementation="eager"`

```python
model = AutoModel.from_pretrained(
    model_name,
    attn_implementation="eager",  # <-- Required for Neuron
    ...
)
```

**Why**: HuggingFace Transformers defaults to `sdpa` (Scaled Dot-Product Attention via `torch.nn.functional.scaled_dot_product_attention`). The SDPA kernel has GPU-specific optimizations (FlashAttention, memory-efficient attention) that don't exist on Neuron. The `eager` implementation uses explicit matrix multiplications (`Q @ K.T`, softmax, `attn @ V`) that Neuron handles natively.

**What happens if you don't**: You may get compilation errors or unexpected behavior, as the SDPA dispatcher tries to select a GPU-optimized attention path that doesn't exist.

### Change #3: `dtype=torch.bfloat16`

```python
model = AutoModel.from_pretrained(
    model_name,
    dtype=torch.bfloat16,  # <-- Neuron's native dtype
    ...
)
```

**Why**: NeuronCores compute natively in bfloat16. While float32 is supported (it gets auto-downcast), loading the model directly in bfloat16:
- Halves memory usage (4B params = ~8GB in bf16 vs ~16GB in fp32)
- Avoids implicit casting overhead
- Matches the model's published dtype (it was trained in bf16)

The Qwen3-Embedding-4B model card specifies `torch_dtype: bfloat16`, so this is the intended precision.

### Change #4: Move inputs to device

```python
inputs = {k: v.to(device) for k, v in batch.items()}
```

**Why**: Just like GPU, all input tensors must be on the same device as the model. If ANY input is on CPU while the model is on Neuron, the operation either fails or (worse) silently falls back to CPU execution.

### Change #5: Pad sequences to multiples of 32

```python
padded_len = math.ceil(seq_len / 32) * 32
batch = tokenizer(texts, padding='max_length', max_length=padded_len, ...)
```

**Why**: We discovered that bfloat16 attention can produce NaN values at certain batch_size/sequence_length combinations (e.g., bs=5, seq_len=27 produces NaN, while bs=5, seq_len=32 is fine). This is a numerical stability issue in the bf16 attention computation at non-aligned shapes.

Padding to multiples of 32 avoids these edge cases entirely. The extra padding tokens are masked out by the attention mask, so they don't affect the embeddings -- they just ensure the internal matrix dimensions are hardware-friendly.

**Note**: This issue does NOT occur in float32, only in bfloat16. If you need exact sequence lengths, you can use float32 attention as a fallback, but bf16 with padded lengths is the recommended approach for production.

### Change #6: Normalize in float32

```python
embeddings = F.normalize(pooled.float(), p=2, dim=1)
```

**Why**: L2 normalization involves computing the sum of squares and a square root. In bfloat16, the limited mantissa precision (7 bits vs 23 in fp32) can introduce small errors in the norm computation. Casting to float32 before normalizing gives more precise unit vectors. The cost is negligible since this is a single vector operation per sentence.

### What Doesn't Need to Change

Everything else works as-is:
- **GQA (Grouped Query Attention)**: Neuron handles the 32Q/8KV head split natively
- **RoPE positional encoding**: Standard rotary embeddings, fully supported
- **RMSNorm**: Supported in eager mode
- **SiLU activation**: Native PyTorch op, works on Neuron
- **Last-token pooling**: Pure tensor indexing, device-agnostic
- **L2 normalization**: `F.normalize` works on any device

---
## Part 4: Running on Neuron

In [6]:
# Verify Neuron device is available
device = torch.device("neuron")
print(f"Target device: {device}")
print(f"PyTorch version: {torch.__version__}")

Target device: neuron
PyTorch version: 2.10.0+cpu


In [7]:
# Load model for Neuron
# Changes from CPU version:
#   1. attn_implementation="eager"  -- avoid SDPA GPU kernels
#   2. torch_dtype=torch.bfloat16   -- Neuron's native compute dtype

import time

print("Loading model...")
t0 = time.time()

neuron_model = AutoModel.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

print(f"Model loaded from disk in {time.time() - t0:.1f}s")
print(f"Model dtype: {next(neuron_model.parameters()).dtype}")

Loading model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded from disk in 0.3s
Model dtype: torch.bfloat16


In [8]:
# Move model to Neuron device
# This transfers all parameters from CPU RAM to NeuronCore HBM
# On trn2.3xlarge with LNC=2, each core has 24GB HBM
# The 4B model in bf16 is ~8GB -- fits easily on a single core

print("Moving model to Neuron...")
t0 = time.time()

neuron_model = neuron_model.to(device)
neuron_model.eval()

print(f"Model moved to Neuron in {time.time() - t0:.1f}s")
print(f"Model device: {next(neuron_model.parameters()).device}")

Moving model to Neuron...


Model moved to Neuron in 8.4s
Model device: neuron:0


In [9]:
# Move inputs to Neuron device
neuron_inputs = {k: v.to(device) for k, v in batch.items()}

# Verify all inputs are on the correct device
for k, v in neuron_inputs.items():
    print(f"  {k}: shape={v.shape}, device={v.device}, dtype={v.dtype}")

  input_ids: shape=torch.Size([5, 32]), device=neuron:0, dtype=torch.int64
  attention_mask: shape=torch.Size([5, 32]), device=neuron:0, dtype=torch.int64


In [10]:
# Generate embeddings on Neuron
# First run triggers NEFF compilation -- this takes several minutes.
# Subsequent runs use the cached NEFF and are much faster.

print("Generating embeddings (first run compiles -- this may take a few minutes)...")
t0 = time.time()

with torch.no_grad():
    outputs = neuron_model(**neuron_inputs)

compile_time = time.time() - t0
print(f"First inference (includes compilation): {compile_time:.1f}s")

# Extract embeddings -- last_token_pool works on any device
# Move attention_mask to same device for the pooling operation
neuron_embeddings = last_token_pool(
    outputs.last_hidden_state,
    neuron_inputs['attention_mask'],
)

# L2 normalize (cast to float32 for precision -- see Change #6)
neuron_embeddings = F.normalize(neuron_embeddings.float(), p=2, dim=1)

print(f"Neuron embedding shape: {neuron_embeddings.shape}")
print(f"Neuron embedding device: {neuron_embeddings.device}")
print(f"Neuron embedding dtype: {neuron_embeddings.dtype}")

Neuron backend does not support int64. Automatically casting to int32. Consider using int32 directly for better performance until native int64 support is added.


Generating embeddings (first run compiles -- this may take a few minutes)...


First inference (includes compilation): 16.2s


Neuron embedding shape: torch.Size([5, 2560])
Neuron embedding device: neuron:0
Neuron embedding dtype: torch.float32


---
## Part 5: Accuracy Validation

The critical question: **do the Neuron embeddings match CPU?**

We compare using cosine similarity. Since both sets of embeddings are L2-normalized, cosine similarity is just the dot product. A score of 1.0 means identical; anything above 0.999 is excellent.

Note: we expect small differences because:
- CPU reference uses float32, Neuron uses bfloat16
- bfloat16 has fewer mantissa bits (7 vs 23), so there's quantization noise
- This is inherent to the dtype, not a Neuron-specific issue

In [11]:
# Compare Neuron embeddings with CPU reference
# Move Neuron embeddings to CPU for comparison
neuron_embeddings_cpu = neuron_embeddings.float().cpu()

# Per-sentence cosine similarity
print("Per-sentence cosine similarity (Neuron vs CPU):")
print("=" * 50)
cosine_sims = []
for i, text in enumerate(all_texts):
    sim = F.cosine_similarity(
        neuron_embeddings_cpu[i:i+1],
        cpu_embeddings_saved[i:i+1],
    ).item()
    cosine_sims.append(sim)
    label = "query" if i < len(queries) else "doc"
    text_preview = text[:60] + "..." if len(text) > 60 else text
    print(f"  [{label} {i}] cos_sim = {sim:.6f}  |  {text_preview}")

avg_sim = sum(cosine_sims) / len(cosine_sims)
min_sim = min(cosine_sims)
print(f"\nAverage cosine similarity: {avg_sim:.6f}")
print(f"Minimum cosine similarity: {min_sim:.6f}")
print(f"\nResult: {'PASS' if min_sim > 0.99 else 'FAIL'} (threshold: 0.99)")

Per-sentence cosine similarity (Neuron vs CPU):
  [query 0] cos_sim = 0.999749  |  Instruct: Given a web search query, retrieve relevant passag...
  [query 1] cos_sim = 0.999762  |  Instruct: Given a web search query, retrieve relevant passag...
  [doc 2] cos_sim = 0.999866  |  Paris is the capital and most populous city of France, situa...
  [doc 3] cos_sim = 0.999793  |  Photosynthesis is the process by which green plants convert ...
  [doc 4] cos_sim = 0.999829  |  The Eiffel Tower was built in 1889 for the World's Fair in P...

Average cosine similarity: 0.999800
Minimum cosine similarity: 0.999749

Result: PASS (threshold: 0.99)


In [12]:
# Verify the retrieval task still works correctly
# Query 0 ("capital of France") should match Doc 0 ("Paris is the capital...")
# Query 1 ("photosynthesis") should match Doc 1 ("Photosynthesis is the process...")

neuron_sim_matrix = neuron_embeddings_cpu[:len(queries)] @ neuron_embeddings_cpu[len(queries):].T

print("Query-Document similarity matrix (Neuron):")
print(neuron_sim_matrix.numpy().round(4))
print("\nRetrieval ranking:")
for i, q in enumerate(queries):
    q_short = q.split("Query: ")[1] if "Query: " in q else q[:50]
    scores = neuron_sim_matrix[i]
    ranked = scores.argsort(descending=True)
    print(f"  Query: '{q_short}'")
    for rank, doc_idx in enumerate(ranked):
        doc_preview = documents[doc_idx][:60]
        print(f"    #{rank+1}: (score={scores[doc_idx]:.4f}) {doc_preview}")

Query-Document similarity matrix (Neuron):
[[0.7005 0.1496 0.4033]
 [0.1224 0.5036 0.1164]]

Retrieval ranking:
  Query: 'What is the capital of France?'
    #1: (score=0.7005) Paris is the capital and most populous city of France, situa
    #2: (score=0.4033) The Eiffel Tower was built in 1889 for the World's Fair in P
    #3: (score=0.1496) Photosynthesis is the process by which green plants convert 
  Query: 'How does photosynthesis work?'
    #1: (score=0.5036) Photosynthesis is the process by which green plants convert 
    #2: (score=0.1224) Paris is the capital and most populous city of France, situa
    #3: (score=0.1164) The Eiffel Tower was built in 1889 for the World's Fair in P


---
## Part 6: Performance Benchmarking

### Understanding Neuron Compilation

PyTorch Native compiles operations into **NEFFs** (Neuron Executable File Format) on first execution. This is similar to `torch.compile` on GPU but happens automatically in eager mode.

- **First run**: Slow (compilation). NEFFs are cached to disk.
- **Subsequent runs**: Fast (cached NEFFs are reused).

The persistent NEFF cache means even restarting the Python process or rebooting the instance reuses compiled NEFFs.

In [13]:
# Warm-up run (uses cached NEFFs from Part 4)
with torch.no_grad():
    _ = neuron_model(**neuron_inputs)
    torch.neuron.synchronize()  # Ensure all async work completes

# Benchmark: measure throughput
num_runs = 20
batch_size = len(all_texts)

torch.neuron.synchronize()
t0 = time.time()

for _ in range(num_runs):
    with torch.no_grad():
        outputs = neuron_model(**neuron_inputs)

torch.neuron.synchronize()
elapsed = time.time() - t0

avg_latency = elapsed / num_runs * 1000  # ms
throughput = (num_runs * batch_size) / elapsed  # sentences/sec

print(f"Benchmark Results ({num_runs} runs, batch_size={batch_size}):")
print(f"  Average latency: {avg_latency:.1f} ms/batch")
print(f"  Throughput: {throughput:.1f} sentences/sec")
print(f"  Per-sentence latency: {avg_latency/batch_size:.1f} ms")

Benchmark Results (20 runs, batch_size=5):
  Average latency: 247.8 ms/batch
  Throughput: 20.2 sentences/sec
  Per-sentence latency: 49.6 ms


### Optional: `torch.compile` with Neuron Backend

PyTorch Native supports `torch.compile` with a Neuron-specific backend. This can fuse operations and reduce host-device round-trips, potentially improving throughput.

**Note**: `torch.compile` with `dynamic=True` is not supported on Neuron. Use static shapes (fixed batch size and sequence length) for best results.

In [14]:
# Compile the model with Neuron backend
print("Compiling model with torch.compile(backend='neuron')...")
t0 = time.time()
compiled_model = torch.compile(neuron_model, backend="neuron")
print(f"torch.compile call returned in {time.time() - t0:.1f}s")

# First inference triggers actual compilation
print("Running first compiled inference (triggers compilation)...")
t0 = time.time()
with torch.no_grad():
    compiled_outputs = compiled_model(**neuron_inputs)
    torch.neuron.synchronize()
print(f"First compiled inference: {time.time() - t0:.1f}s")

# Warm up
for _ in range(3):
    with torch.no_grad():
        _ = compiled_model(**neuron_inputs)
torch.neuron.synchronize()

# Benchmark compiled model
torch.neuron.synchronize()
t0 = time.time()
for _ in range(num_runs):
    with torch.no_grad():
        outputs = compiled_model(**neuron_inputs)
torch.neuron.synchronize()
elapsed = time.time() - t0

compiled_latency = elapsed / num_runs * 1000
compiled_throughput = (num_runs * batch_size) / elapsed

print(f"\nCompiled Model Benchmark ({num_runs} runs, batch_size={batch_size}):")
print(f"  Average latency: {compiled_latency:.1f} ms/batch")
print(f"  Throughput: {compiled_throughput:.1f} sentences/sec")
print(f"  Speedup vs eager: {avg_latency/compiled_latency:.2f}x")

Compiling model with torch.compile(backend='neuron')...
torch.compile call returned in 0.0s
Running first compiled inference (triggers compilation)...


First compiled inference: 64.9s



Compiled Model Benchmark (20 runs, batch_size=5):
  Average latency: 35.0 ms/batch
  Throughput: 143.0 sentences/sec
  Speedup vs eager: 7.08x


---
## Part 7: Summary

### Changes Required for Neuron

| # | Change | Why | Impact if Skipped |
|---|--------|-----|-------------------|
| 1 | `device = torch.device("neuron")` | Use NeuronCore hardware | Runs on CPU (no acceleration) |
| 2 | `attn_implementation="eager"` | Avoid GPU-specific SDPA kernels | Compilation errors |
| 3 | `dtype=torch.bfloat16` | Native Neuron compute type | Works in fp32 but wastes HBM and may be slower |
| 4 | `inputs.to(device)` | All tensors must be on same device | Silent CPU fallback or errors |
| 5 | Pad sequences to multiples of 32 | Avoid bf16 NaN at non-aligned shapes | NaN embeddings at certain batch/seq_len combos |
| 6 | `F.normalize(x.float(), ...)` | Precise L2 norm in float32 | Slightly less precise unit vectors |

### What Works Out-of-the-Box

- GQA (Grouped Query Attention) with 4:1 head ratio
- RoPE positional encoding
- RMSNorm layer normalization
- SiLU activation
- Last-token pooling (pure tensor indexing)
- L2 normalization
- HuggingFace Transformers model loading

### Key Takeaways

1. **PyTorch Native makes Neuron feel like GPU**: The API is identical -- `model.to(device)`, `tensor.to(device)`, standard PyTorch ops. No XLA graph tracing or special compilation steps.

2. **First-run compilation is the cost**: The first inference triggers NEFF compilation (minutes). After that, cached NEFFs make subsequent runs fast. This cache persists across process restarts.

3. **bfloat16 is your friend**: It's Neuron's native dtype, halves memory, and the model was trained in it. The tiny accuracy difference vs fp32 is inherent to the dtype, not to Neuron.

4. **Pad to multiples of 32**: Certain batch_size/seq_len combinations can produce NaN in bf16 attention. Padding to multiples of 32 eliminates this with negligible overhead (masked tokens don't affect output).

5. **torch.compile gives ~7x speedup**: The Neuron backend for torch.compile fuses operations aggressively, delivering major throughput gains over eager mode.

6. **Decoder-only embeddings are straightforward on Neuron**: No special embedding-specific changes needed. The model is just a standard causal LM; the embedding magic is in post-processing (last-token pool + normalize).

### Measured Results (trn2.3xlarge, bs=5, seq_len=32)

| Metric | Value |
|--------|-------|
| Accuracy (avg cosine sim vs CPU fp32) | 0.9998 |
| Accuracy (min cosine sim) | 0.9997 |
| Eager throughput | 20.5 sentences/sec |
| Compiled throughput | 143.4 sentences/sec |
| torch.compile speedup | 6.98x |

In [15]:
# Final summary
print("=" * 60)
print("Qwen3-Embedding-4B on Neuron -- Results Summary")
print("=" * 60)
print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Dtype: bfloat16")
print(f"Batch size: {batch_size}")
print(f"")
print(f"Accuracy:")
print(f"  Avg cosine sim (vs CPU fp32): {avg_sim:.6f}")
print(f"  Min cosine sim (vs CPU fp32): {min_sim:.6f}")
print(f"")
print(f"Performance (eager):")
print(f"  Latency: {avg_latency:.1f} ms/batch")
print(f"  Throughput: {throughput:.1f} sentences/sec")
print(f"")
print(f"Performance (torch.compile):")
print(f"  Latency: {compiled_latency:.1f} ms/batch")
print(f"  Throughput: {compiled_throughput:.1f} sentences/sec")
print(f"  Speedup: {avg_latency/compiled_latency:.2f}x")

Qwen3-Embedding-4B on Neuron -- Results Summary
Model: Qwen/Qwen3-Embedding-4B
Device: neuron
Dtype: bfloat16
Batch size: 5

Accuracy:
  Avg cosine sim (vs CPU fp32): 0.999800
  Min cosine sim (vs CPU fp32): 0.999749

Performance (eager):
  Latency: 247.8 ms/batch
  Throughput: 20.2 sentences/sec

Performance (torch.compile):
  Latency: 35.0 ms/batch
  Throughput: 143.0 sentences/sec
  Speedup: 7.08x
